In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  

ARQUIVO = 'Tabela_13_Meio_de_transporte.xlsx'
RACAS = ['Branca', 'Preta ou Parda', 'Indígena']
MODOS = ['A pé', 'Bicicleta', 'Motocicleta ou Mototáxi', 'Automóvel, táxi ou assemelhados', 'Transporte coletivo', 'Outros']
CORES = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']
xls = pd.ExcelFile(ARQUIVO)
aba_ex = next(s for s in xls.sheet_names if 'Exemplo' in s)
aba2 = next(s for s in xls.sheet_names if 'BR GR UF MU' in s)
print('Abas:', xls.sheet_names)


In [ ]:
df = pd.read_excel(ARQUIVO, sheet_name=aba_ex, header=1).iloc[:len(MODOS)].copy()
df.columns = ['Meio de Transporte', *RACAS]
df['Meio de Transporte'] = MODOS  
for raca in RACAS:
    df[raca] = pd.to_numeric(df[raca], errors='coerce')
df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, col in enumerate(RACAS):
    axes[i].pie(df[col], labels=df['Meio de Transporte'], autopct='%1.3f%%', pctdistance=0.78,
                startangle=180, colors=CORES,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.2})
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')
plt.suptitle('Meio de transporte no deslocamento ao trabalho, por cor ou raça — Brasil, 2022',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
raw = pd.read_excel(ARQUIVO, sheet_name=aba2, header=None)
ESTADOS = ['Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins',
           'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco',
           'Alagoas', 'Sergipe', 'Bahia', 'Minas Gerais', 'Espírito Santo',
           'Rio de Janeiro', 'São Paulo', 'Paraná', 'Santa Catarina',
           'Rio Grande do Sul', 'Mato Grosso do Sul', 'Mato Grosso', 'Goiás', 'Distrito Federal']
raw.iloc[9:36, 0] = ESTADOS
MODOS = ['A pé', 'Bicicleta', 'Motocicleta ou Mototáxi',
         'Automóvel, táxi ou assemelhados', 'Transporte coletivo', 'Outros']
lin = raw.iloc[3:].copy()
lin = lin[lin.iloc[:, 0].notna()]

pts = []
for i, raca in enumerate(RACAS):
    tmp = lin.iloc[:, [0, *range(1 + i*6, 1 + (i+1)*6)]].copy()
    tmp.columns = ['Local', *MODOS]
    tmp = tmp.melt(id_vars='Local', var_name='Meio de Transporte', value_name='Percentual')
    tmp['Cor/Raça'] = raca
    pts.append(tmp)
dl = pd.concat(pts, ignore_index=True)
dl['Local'] = dl['Local'].astype(str).str.strip()
dl['Percentual'] = pd.to_numeric(dl['Percentual'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
dl.head()


In [ ]:

ufs = raw.iloc[9:36, 0].dropna().astype(str).str.strip().tolist()
mun = raw.iloc[36:, 0].dropna().astype(str).str.strip().tolist()
uf_df = dl[dl['Local'].isin(ufs)].copy()
uf_df[['Local']].drop_duplicates().sort_values('Local').reset_index(drop=True)


In [ ]:
ger = (dl.groupby(['Local', 'Meio de Transporte'], as_index=False)['Percentual']
         .mean())  
uf_g = ger[ger['Local'].isin(ufs)]
mun_g = ger[ger['Local'].isin(mun)]
coletivo_est = uf_g[uf_g['Meio de Transporte'].eq('Transporte coletivo')].dropna(subset=['Percentual'])
print('Maior transporte coletivo:', coletivo_est.loc[coletivo_est['Percentual'].idxmax()].to_dict())
print('Menor transporte coletivo:', coletivo_est.loc[coletivo_est['Percentual'].idxmin()].to_dict())
print('Maior bicicleta:', mun_g[mun_g['Meio de Transporte'].eq('Bicicleta')].dropna(subset=['Percentual']).nlargest(1, 'Percentual'))
print('Maior automóvel/táxi:', mun_g[mun_g['Meio de Transporte'].eq('Automóvel, táxi ou assemelhados')].dropna(subset=['Percentual']).nlargest(1, 'Percentual'))




 Maior percentual médio de mulheres no transporte coletivo entre os estados: **Rio de Janeiro (52.23%)**.
 Menor percentual médio: **Rondônia (4.33%)**.
 Cidade com maior percentual médio usando bicicleta: **Tuiuti (SP) (100.00%)**.
 Cidade com maior percentual médio usando automóvel/táxi: **Bom Jesus do Oeste (SC) (86.05%)**.

